# Style-Bert-VITS2 吹き替え自動化ツール

### 1. フォルダ構成
以下のようにファイルを配置してください：

```
Style-Bert-VITS2/
├── input_mp4/
│   ├── srt/                    # ← 全ての字幕ファイル(.srt)をここに配置
│   │   ├── 動画1.srt
│   │   ├── 動画2.srt
│   │   └── ...
│   ├── 講座A/                  # ← 動画ファイル(.mp4)はフォルダ分けしてOK
│   │   ├── 動画1.mp4
│   │   └── 動画2.mp4
│   └── 講座B/
│       └── 動画3.mp4
└── output_mp4/                 # ← 吹き替え後の動画がここに出力される
    ├── 講座A/
    │   ├── 動画1.mp4         # 吹き替え済み
    │   ├── 動画1.srt         # 字幕も一緒にコピー
    │   ├── 動画2.mp4
    │   └── 動画2.srt
    └── 講座B/
        ├── 動画3.mp4
        └── 動画3.srt
```

### 2. モデルの準備
- `model_assets/` フォルダ内に使用したいTTSモデルを配置
- 以下のコードで `model_name="モデル名"` を指定するだけでOK

### 3. 字幕ファイルの形式
- SRTファイル名は動画ファイル名と同じにしてください
  - 例: `動画1.mp4` → `動画1.srt`
- 文字コードは自動検出（UTF-8, Shift_JIS, CP932対応）
- 出力時は **UTF-8 BOM付き** で保存されます（Windowsメディアプレイヤーで文字化けしません）

## ⚙️ 機能

✅ **音声カット禁止ポリシー**
- 字幕の時間内に収まらない場合は話速を自動調整（最大2倍速まで）

✅ **英語→カタカナ自動変換**
- 22万語の辞書による高精度変換
- 辞書にない単語は eng_to_ipa でフォールバック

✅ **イントロ音声重ね合わせ**
- 最初の字幕が始まるまでは元動画の音声を残します（フェードアウト付き）

✅ **重複処理スキップ**
- 既に出力済みの動画は上書きせずスキップ

## 🚀 使い方

Cell 2を実行してください。

In [ ]:
import sys
import os
from pathlib import Path

# 親ディレクトリをパスに追加(dubbing_toolsをインポートできるようにする)
parent_dir = str(Path(__file__).parent.parent) if '__file__' in globals() else str(Path.cwd().parent)
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)

from dubbing_tools import DubbingAutomation

# モデル初期化(モデル名だけで自動的にファイルを検索)
dubbing = DubbingAutomation(
    model_name="ui_speaker",  # model_assets/モデル名/ 内のファイルを自動検索
    device="cuda",
)

# input_mp4内のすべてのMP4を自動検索
input_base = os.path.join(parent_dir, "input_mp4")
output_base = os.path.join(parent_dir, "output_mp4")
srt_folder = os.path.join(input_base, "srt")

for root, dirs, files in os.walk(input_base):
    if "srt" in root:  # srtフォルダはスキップ
        continue
        
    for file in files:
        if file.endswith('.mp4'):
            video_path = os.path.join(root, file)
            name = file.rsplit('.', 1)[0]
            srt_path = os.path.join(srt_folder, f"{name}.srt")
            
            if os.path.exists(srt_path):
                # 出力先: input_mp4/講座A → output_mp4/講座A
                relative_path = os.path.relpath(video_path, input_base)
                output_path = os.path.join(output_base, relative_path)
                
                # 既に出力済みの場合はスキップ
                if os.path.exists(output_path):
                    print(f"\nスキップ(既に存在): {output_path}")
                    continue
                
                print(f"\n処理中: {video_path}")
                dubbing.create_dubbed_video(
                    video_path=video_path,
                    srt_path=srt_path,
                    output_path=output_path,
                    intro_only=True,  # イントロ部分のみ元の音声を重ねる
                    overlay=True,
                    audio_volume=1.0,
                    original_volume=0.3,
                )
                print(f"完了: {output_path}")

TTSモデルを読み込み中: ui_speaker
  モデルファイル: c:\Users\Phant\Documents\ui_kaihatu\Style-Bert-VITS2\model_assets\ui_speaker\ui_speaker.safetensors
TTSモデル読み込み完了
[OK] 読み方調整を3件読み込みました: c:\Users\Phant\Documents\ui_kaihatu\Style-Bert-VITS2\dubbing_tools\reading_adjustments.csv
[OK] 英単語辞書を221618件読み込みました: c:\Users\Phant\Documents\ui_kaihatu\Style-Bert-VITS2\dubbing_tools\english_katakana_dict.csv

処理中: c:\Users\Phant\Documents\ui_kaihatu\Style-Bert-VITS2\input_mp4\[字幕なし]完全攻略60講：最速で実力を高めるカジュアルイラスト\01.エキナ\[エキナ]Section 01.自分が描きたいものを考える\01.絵を描く前の心構え.mp4
SRTファイルを解析中: c:\Users\Phant\Documents\ui_kaihatu\Style-Bert-VITS2\input_mp4\srt\01.絵を描く前の心構え.srt
SRTファイルのエンコーディング: utf-8-sig
253個の字幕エントリを検出
動画の長さを取得中: c:\Users\Phant\Documents\ui_kaihatu\Style-Bert-VITS2\input_mp4\[字幕なし]完全攻略60講：最速で実力を高めるカジュアルイラスト\01.エキナ\[エキナ]Section 01.自分が描きたいものを考える\01.絵を描く前の心構え.mp4
動画の長さ: 929.10秒
音声を生成中...
11-25 18:27:54 |  INFO  | tts_model.py:410 | Start generating audio data from text:
こんにちは
動画の長さ: 929.10秒
音声を生成中...
11-25 18:27:54 |  INF

c:\Users\Phant\Documents\ui_kaihatu\Style-Bert-VITS2\venv\lib\site-packages\torch\nn\utils\weight_norm.py:143: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


11-25 18:27:57 |  INFO  | safetensors.py:51 | Loaded 'c:\Users\Phant\Documents\ui_kaihatu\Style-Bert-VITS2\model_assets\ui_speaker\ui_speaker.safetensors' (iteration 100)
11-25 18:27:57 |  INFO  | tts_model.py:152 | Model loaded successfully from c:\Users\Phant\Documents\ui_kaihatu\Style-Bert-VITS2\model_assets\ui_speaker\ui_speaker.safetensors to "cuda" device (0.67s)
11-25 18:27:57 |  INFO  | tts_model.py:152 | Model loaded successfully from c:\Users\Phant\Documents\ui_kaihatu\Style-Bert-VITS2\model_assets\ui_speaker\ui_speaker.safetensors to "cuda" device (0.67s)


c:\Users\Phant\Documents\ui_kaihatu\Style-Bert-VITS2\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


11-25 18:27:59 |  INFO  | bert_models.py:180 | Loaded the JP BERT tokenizer from c:\Users\Phant\Documents\ui_kaihatu\Style-Bert-VITS2\bert\deberta-v2-large-japanese-char-wwm
11-25 18:28:00 |  INFO  | bert_models.py:114 | Loaded the JP BERT model from c:\Users\Phant\Documents\ui_kaihatu\Style-Bert-VITS2\bert\deberta-v2-large-japanese-char-wwm (0.51s)
11-25 18:28:00 |  INFO  | bert_models.py:114 | Loaded the JP BERT model from c:\Users\Phant\Documents\ui_kaihatu\Style-Bert-VITS2\bert\deberta-v2-large-japanese-char-wwm (0.51s)
11-25 18:28:00 |  INFO  | tts_model.py:557 | Audio data generated successfully (5.87s)
  [2] テキスト変換(英語→カタカナ): 今回Colosoで講座を行う → 今回コロソで講座を行う
11-25 18:28:00 |  INFO  | tts_model.py:410 | Start generating audio data from text:
今回コロソで講座を行う
11-25 18:28:00 |  INFO  | tts_model.py:557 | Audio data generated successfully (5.87s)
  [2] テキスト変換(英語→カタカナ): 今回Colosoで講座を行う → 今回コロソで講座を行う
11-25 18:28:00 |  INFO  | tts_model.py:410 | Start generating audio data from text:
今回コロソで講座を行う


In [1]:
# カタカナ英語変換テスト
import sys
from pathlib import Path

# 親ディレクトリをパスに追加
parent_dir = str(Path.cwd().parent)
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)

from dubbing_tools.text_preprocessor import TextPreprocessor

# TextPreprocessorを再初期化（ライブラリインストール後）
preprocessor = TextPreprocessor()

# テストケース
test_words = [
    "coloso",      # 辞書にある（loanwords_gairaigo統合済み）
    "supercalifragilisticexpialidocious",  # 辞書にない（eng_to_ipaでテスト）
    "xylophone",   # 辞書にない可能性
    "photoshop",   # 辞書にある
]

print("=== 英語→カタカナ変換テスト ===\n")
for word in test_words:
    result = preprocessor.convert_english_to_katakana(word)
    print(f"{word:40} → {result}")

[OK] 読み方調整を3件読み込みました: c:\Users\Phant\Documents\ui_kaihatu\Style-Bert-VITS2\dubbing_tools\reading_adjustments.csv
[OK] 英単語辞書を221589件読み込みました: c:\Users\Phant\Documents\ui_kaihatu\Style-Bert-VITS2\dubbing_tools\english_katakana_dict.csv
=== 英語→カタカナ変換テスト ===

[Web API] coloso → コーローソー (辞書に登録)
coloso                                   → コーローソー
supercalifragilisticexpialidocious       → スーパーカリフラジリスティックエクスピアリドーシャス
xylophone                                → シロフォン
photoshop                                → フォトショップ
[Web API] coloso → コーローソー (辞書に登録)
coloso                                   → コーローソー
supercalifragilisticexpialidocious       → スーパーカリフラジリスティックエクスピアリドーシャス
xylophone                                → シロフォン
photoshop                                → フォトショップ


In [ ]:
# ローマ字→カタカナ変換テスト
from text_preprocessor import TextPreprocessor

# テストケース
test_romaji = [
    "konnichiwa",
    "arigatou",
    "sayonara",
    "ohayou",
    "kawaii",
    "subarashii",
    "ganbatte",
    "toukyou",
    "kyouto",
    "nihon",
    "senpai",
    "kouhai",
]

print("=== ローマ字→カタカナ変換テスト ===\n")
for romaji in test_romaji:
    katakana = TextPreprocessor.romaji_to_katakana(romaji)
    print(f"{romaji:30} → {katakana}")

ModuleNotFoundError: No module named 'dubbing_tools'

In [2]:
import csv
import sqlite3
from pathlib import Path

# loanwords_gairaigo のSQLファイルから辞書を抽出
sql_file = r'C:\Users\Phant\Documents\ui_kaihatu\Style-Bert-VITS2\loanwords_gairaigo-master\loanwords_gairaigo\db\merged.sql'
temp_db = Path.cwd() / "temp_loanwords.db"

print(f"SQLファイル: {sql_file}")
print("データベースを構築中...")

# SQLファイルを実行して一時DBを作成
conn = sqlite3.connect(temp_db)
cursor = conn.cursor()

try:
    with open(sql_file, 'r', encoding='utf-8') as f:
        sql_script = f.read()
        cursor.executescript(sql_script)
    
    print("✅ データベース構築完了")
    
    # テーブル一覧を確認
    cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
    tables = cursor.fetchall()
    print(f"\nテーブル一覧: {[t[0] for t in tables]}")
    
    # 最初のテーブルの構造を確認
    if tables:
        table_name = tables[0][0]
        cursor.execute(f"PRAGMA table_info({table_name})")
        columns = cursor.fetchall()
        print(f"\n{table_name} テーブルの列:")
        for col in columns:
            print(f"  {col[1]} ({col[2]})")
        
        # データを取得
        cursor.execute(f"SELECT * FROM {table_name} LIMIT 10")
        sample_data = cursor.fetchall()
        print(f"\nサンプルデータ（最初の10件）:")
        for row in sample_data:
            print(f"  {row}")
        
        # 全データを取得してCSVに追加
        cursor.execute(f"SELECT * FROM {table_name}")
        all_data = cursor.fetchall()
        print(f"\n総データ数: {len(all_data)}件")
        
        # english_katakana_dict.csvに追記
        csv_path = Path.cwd() / "english_katakana_dict.csv"
        
        # 既存の辞書を読み込んで重複チェック
        existing_words = set()
        if csv_path.exists():
            with open(csv_path, 'r', encoding='utf-8-sig') as f:
                reader = csv.reader(f)
                for row in reader:
                    if row and not row[0].startswith('#'):
                        existing_words.add(row[0].lower())
        
        # 新規データのみ追加
        added_count = 0
        with open(csv_path, 'a', encoding='utf-8-sig', newline='') as f:
            writer = csv.writer(f)
            for row in all_data:
                # SQLの列構造に応じて調整（通常は english, katakana の順）
                if len(row) >= 2:
                    english = str(row[0]).strip().lower()
                    katakana = str(row[1]).strip()
                    
                    if english and katakana and english not in existing_words:
                        writer.writerow([english, katakana])
                        added_count += 1
                        existing_words.add(english)
                        
                        if added_count <= 20:  # 最初の20件だけ表示
                            print(f"追加: {english} → {katakana}")
        
        print(f"\n完了: {added_count}件の新規単語を追加しました")
        print(f"(重複スキップ: {len(all_data) - added_count}件)")
        
except Exception as e:
    print(f"❌ エラー: {e}")
    import traceback
    traceback.print_exc()
finally:
    conn.close()
    # 一時DBを削除
    if temp_db.exists():
        temp_db.unlink()
        print("\n一時データベースを削除しました")

SQLファイル: C:\Users\Phant\Documents\ui_kaihatu\Style-Bert-VITS2\loanwords_gairaigo-master\loanwords_gairaigo\db\merged.sql
データベースを構築中...
✅ データベース構築完了

テーブル一覧: ['merged']

merged テーブルの列:
  english (TEXT)
  japanese (TEXT)

サンプルデータ（最初の10件）:
  ("'ALI-SULTAN", 'アリースルターン')
  ("'NDRANGHETA", 'ンドランゲタ')
  ("'PATAPHYSICS", 'パタフィジック')
  ("'S-HERTOGENBOSCH", 'スヘルトーヘンボス')
  ("'ULI'ULI", 'ウリウリ')
  ('A', 'エー')
  ('AGONIST', 'アゴニスト')
  ('AIDS', 'エイズ')
  ('ALUMINA', 'アルミナ')
  ('ALUMINUM', 'アルミニウム')

総データ数: 221588件
✅ データベース構築完了

テーブル一覧: ['merged']

merged テーブルの列:
  english (TEXT)
  japanese (TEXT)

サンプルデータ（最初の10件）:
  ("'ALI-SULTAN", 'アリースルターン')
  ("'NDRANGHETA", 'ンドランゲタ')
  ("'PATAPHYSICS", 'パタフィジック')
  ("'S-HERTOGENBOSCH", 'スヘルトーヘンボス')
  ("'ULI'ULI", 'ウリウリ')
  ('A', 'エー')
  ('AGONIST', 'アゴニスト')
  ('AIDS', 'エイズ')
  ('ALUMINA', 'アルミナ')
  ('ALUMINUM', 'アルミニウム')

総データ数: 221588件

完了: 0件の新規単語を追加しました
(重複スキップ: 221588件)

一時データベースを削除しました

完了: 0件の新規単語を追加しました
(重複スキップ: 221588件)

一時データベースを削除しました
